# Lab 1 - Serial data acquisition
Reads the CSV stream from the ESP32 sensor node and saves >= 600 rows to `data/raw/`.
Every row is kept, including `INVALID` ones. Comment lines and malformed lines go to a `.log` file.
This notebook should be run from the `python/` folder, but after closing the Arduino Serial Monitor first (only one program can hold the port).

In [1]:
from datetime import datetime
from pathlib import Path
import csv
from serial_utils import COLUMNS, open_port, stream_rows

PORT = "COM4"          # Linux: /dev/ttyUSB0   macOS: /dev/cu.usbserial-xxxx
BAUD = 115200
MIN_ROWS = 600         # lab requirement (10 min at 1 Hz)
RAW_DIR = Path.cwd().parent / "data" / "raw" if Path.cwd().name == "python" else Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = RAW_DIR / f"sensor_log_{stamp}.csv"
out_log = RAW_DIR / f"sensor_log_{stamp}.log"

ser = open_port(PORT, BAUD)
n = 0
with open(out_csv, "w", newline="") as fh, open(out_log, "w") as log:
    w = csv.writer(fh)
    w.writerow(COLUMNS + ["pc_time"])
    try:
        for row in stream_rows(ser, log):
            w.writerow([row[c] for c in COLUMNS] + [datetime.now().strftime("%H:%M:%S")])
            n += 1
            if n % 30 == 0:
                print(f"{n} rows  (last status: {row['status']})", end="\r")
            if n >= MIN_ROWS:
                break
    except KeyboardInterrupt:
        print("\nStopped early by user.")
ser.close()
print(f"\nSaved {n} rows to {out_csv}")

600 rows  (last status: MOTION)
Saved 600 rows to D:\AAAAAA YEAR 3 2026_27\WIRELESS SENSOR NETWORKS\LAB\Lab 1 files\lab1\data\raw\sensor_log_20260929_171534.csv
